[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S13/2_Practica/reto_expres/Reto_expres_3_busqueda.ipynb)

# Reto exprés S12-13 · Embeddings: ¿qué reseñas se parecen?

**Solo se tocan las celdas ✏️** (cambiar un valor y pulsar ▶). Las ⚙️ se ejecutan tal cual.

- ✏️ **1** · probar cuántos números (dimensiones) usa cada reseña y compararlo con **no comprimir**, **mirando solo la validación**
- ✏️ **2** · contadlo: titular con cifra y gráfico de «precisión frente a dimensiones»
- ✏️ **3** · la aplicación: buscar reseñas parecidas a una frase vuestra
- Al final sale una **FICHA** con todos los números para vuestra presentación.

| Qué se hace | Datos | Rival simple | Métrica |
|---|---|---|---|
| encontrar las 3 reseñas más parecidas a una dada | `resenas.csv` (con la columna `temas`) | TF-IDF **sin comprimir** | % de los 3 vecinos con los **mismos temas** (más alto = mejor) |

### ⚙️ Preparación — pulsa ▶

In [ ]:
#@title ⚙️ Ejecuta esta celda primero
import os, re, urllib.request, time
BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"
def dato(nombre):
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta): return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import numpy as np, pandas as pd, matplotlib.pyplot as plt
SEMILLA = 42

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
print("Listo ✔")

### ⚙️ Preparar las reseñas — pulsa ▶

**Corpus** = las reseñas donde se busca; **validación** y **prueba** = reseñas nuevas que se usan como pregunta (la prueba se mira una sola vez). Un vecino **acierta** si habla de los mismos temas que la pregunta.

In [ ]:
df = pd.read_csv(dato("resenas.csv")).dropna(subset=["temas"]).reset_index(drop=True)
corpus, resto = train_test_split(df, test_size=0.4, random_state=SEMILLA)
val, prueba = train_test_split(resto, test_size=0.5, random_state=SEMILLA)
tfidf = TfidfVectorizer(strip_accents="unicode", min_df=2).fit(corpus.texto)
M_corpus = tfidf.transform(corpus.texto)
temas = lambda d: [frozenset(t.split(",")) for t in d.temas]
T_corpus = temas(corpus)

def precision3(codificar, consultas):
    """% de los 3 vecinos más parecidos que hablan de los MISMOS temas que la consulta."""
    sim = cosine_similarity(codificar(tfidf.transform(consultas.texto)), codificar(M_corpus))
    top = np.argsort(-sim, axis=1)[:, :3]
    return float(np.mean([[T_corpus[j] == t for j in fila] for fila, t in zip(top, temas(consultas))]))

sin_comprimir = lambda M: M
def lsa(k):
    svd = TruncatedSVD(k, random_state=SEMILLA).fit(M_corpus); return svd.transform
print(f"{len(corpus)} reseñas en el corpus · {len(val)} de validación · {len(prueba)} de prueba · vocabulario de {M_corpus.shape[1]} palabras")
print(f"Ejemplo: «{val.texto.iloc[0][:100]}» → temas: {val.temas.iloc[0]}")

### ✏️ 1 · Probar dimensiones y elegir (solo con la validación)

Cambiad **DIMENSIONES** (mínimo 3 valores, entre 2 y 50). Mirad **val_precision3** (**más alto = mejor**) y comparadlo con el rival, que **no comprime** (usa una columna por palabra). Poned **ELEGIDA = 0** si os quedáis con el rival.

In [ ]:
DIMENSIONES = [5, 10, 20]
RIVAL_VAL = precision3(sin_comprimir, val)
filas = [dict(candidata="TF-IDF sin comprimir (rival)", numeros_por_resena=M_corpus.shape[1], val_precision3=round(RIVAL_VAL, 3))]
for k in DIMENSIONES:
    filas.append(dict(candidata=f"LSA con {k} dimensiones", numeros_por_resena=k, val_precision3=round(precision3(lsa(k), val), 3)))
tabla = pd.DataFrame(filas); display(tabla)
ELEGIDA = 20               # ✏️ dimensiones elegidas (o 0 para el rival sin comprimir)

### ⚙️ La prueba, una sola vez — pulsa ▶

Comprimir ahorra memoria y puede quitar ruido, pero **no siempre mejora**. Mirad también los fallos: ¿un vecino con otros temas es un error de verdad o es una reseña que también habla de lo mismo?

In [ ]:
codificar = lsa(ELEGIDA) if ELEGIDA > 0 else sin_comprimir
p_elegida = precision3(codificar, prueba); p_rival = precision3(sin_comprimir, prueba)
sim = cosine_similarity(codificar(tfidf.transform(prueba.texto)), codificar(M_corpus)); top1 = sim.argmax(1)
mal = [i for i in range(len(prueba)) if T_corpus[top1[i]] != frozenset(prueba.temas.iloc[i].split(","))]
print(f"Con el primer vecino, {len(mal)} de {len(prueba)} reseñas de prueba no coinciden en temas. Algunos ejemplos:")
for i in mal[:3]:
    print(f"  pregunta «{prueba.texto.iloc[i][:70]}» ({prueba.temas.iloc[i]})\n   vecino   «{corpus.texto.iloc[top1[i]][:70]}» ({corpus.temas.iloc[top1[i]]})")
FICHA = dict(elegida=ELEGIDA, vocabulario=int(M_corpus.shape[1]), rival=round(p_rival, 3), elegida_prueba=round(p_elegida, 3), fallos_vecino1=len(mal), consultas=len(prueba))
print(f"""
================ FICHA PARA LA PRESENTACIÓN ================
Problema ........ buscar las 3 reseñas más parecidas · corpus de {len(corpus)} reseñas
Candidatas ...... {[(f['candidata'], f['val_precision3']) for f in filas]}  (precisión@3 en validación)
Elegida ......... {"LSA con %d dimensiones" % ELEGIDA if ELEGIDA > 0 else "TF-IDF sin comprimir"}
--- PRUEBA (una sola vez) ---
Rival (sin comprimir) ... {p_rival:.1%} de los vecinos con los mismos temas
Elegida ................. {p_elegida:.1%}   {"(compensa)" if p_elegida > p_rival + 0.01 else "(NO mejora al rival: comprimir solo ahorra memoria)"}
============================================================
""")

### ✏️ 2 · Contadlo (S13): titular con cifra y gráfico

Cambiad **TITULAR**. El gráfico enseña **cuánto se pierde al comprimir** (línea = el rival sin comprimir; eje desde 0).
**Cómo se cuenta (S13):** el titular es una **conclusión con cifra** (no «Resultados»), el gráfico empieza en **0** y las cifras salen de variables (f-string), no se teclean a mano. Terminad con un **límite**: ¿cuándo no os fiaríais?

In [ ]:
TITULAR = f"Con {ELEGIDA or M_corpus.shape[1]} números por reseña, {p_elegida:.0%} de los vecinos hablan de lo mismo; sin comprimir, {p_rival:.0%}"   # ✏️ vuestra conclusión
ks = [2, 5, 10, 20, 50]; ps = [precision3(lsa(k), prueba) for k in ks]
fig, ax = plt.subplots(figsize=(7, 3.8)); ax.bar([str(k) for k in ks], ps, color="tab:blue"); ax.axhline(p_rival, color="tab:red", ls="--", label=f"sin comprimir ({p_rival:.0%})")
ax.set_ylim(0, 1); ax.set_xlabel("dimensiones (números por reseña)"); ax.set_ylabel("vecinos con los mismos temas"); ax.legend(); fig.suptitle(TITULAR, fontsize=10, fontweight="bold"); plt.tight_layout(); plt.show()
FICHA["titular"] = TITULAR

### ✏️ 3 · La aplicación: buscar reseñas parecidas

Escribid una frase dentro de `buscar("...")`. Probad con una queja (ruido, precio, limpieza…) y con una frase que mezcle dos temas.

In [ ]:
def buscar(frase, n=3):
    s = cosine_similarity(codificar(tfidf.transform([frase])), codificar(M_corpus))[0]
    print(f"«{frase}»")
    for j in np.argsort(-s)[:n]: print(f"  {s[j]:.2f} · «{corpus.texto.iloc[j][:90]}» ({corpus.temas.iloc[j]})")

buscar("No pudimos dormir por el ruido de la calle"); buscar("escribe aquí tu propia frase")

### 🎤 Para la presentación (2–3 min)

Copiad la **FICHA**, la tabla de la celda ✏️ 1 y el gráfico. Una frase para cada pregunta: **¿compensa comprimir? · ¿por qué esas dimensiones? · ¿qué reseñas confunde? · ¿lo usaríamos de verdad?**